# Stage 2 — Stratified 70/30 train/test split (PD vs HC, stratum-aware VST)

Input: `PDHC new/Outputs/VST/vst_stratum_aware_PDHC.csv` (155 samples).

**Strong control on the `cell_type` confounder.** Sorted-cell RNA-seq has
PBMC / CD4 / CD8 sub-populations with very different mean expression — the
PCA scripts already show `cell_type` dominates PC1-5. So we:

1. Stratify primarily on (`condition` × `cell_type`), composite-augmented with
   `sex` when every sub-stratum has n ≥ 2 (avoids singleton failures).
2. Walk a fallback chain only when composite singletons appear:
   `(condition × cell_type × sex) → (condition × cell_type) → condition`.
3. Run an **explicit per-`cell_type` chi-squared check** between train and
   test condition distributions in addition to the marginal chi-squared,
   so a sex/cell-type imbalance can't hide behind the composite test.
4. Mirror-OLS residualize on (`sex` + `cell_type`) using TRAIN-fit betas only,
   and quantify the per-gene partial R² before vs after to confirm the
   `cell_type` signal really collapses to ≈ 0%.

All diagnostics (chi-squared marginal + per-cell_type, KS on train-fit PCs,
variance-partitioning) are post-hoc — they **never** feed back into the
split.

In [1]:
"""
Stage 2 — Stratified 70/30 split for PD vs HC on the stratum-aware VST.

Strict no-leakage rules
-----------------------
* PCA fit on X_train only; test scores via .transform().
* OLS confounder regression fit on X_train only; test residuals
  computed from the same train-fit betas (one-hot encoder also
  fitted on train metadata only).
* Chi-squared and KS tests are post-hoc diagnostics; they do not
  feed back into the split.
* Test residualized data uses TRAIN-fit betas — never refit on test.
"""

import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR    = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
PDHC_DIR        = PIPELINE_DIR / "PDHC new"
INPUT_VST       = PDHC_DIR / "Outputs" / "VST" / "vst_stratum_aware_PDHC.csv"
METADATA_PATH   = PIPELINE_DIR / "PDvsHC" / "Meta_data_PDHC.csv"
OUTPUT_DIR      = PDHC_DIR / "Outputs" / "stage2_split_stratum_aware_PDHC"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED            = 42
TEST_SIZE       = 0.30
CONDITION_COL   = "condition"
PRIMARY_CONFOUNDER  = "cell_type"   # stronger control: anchor of the split
SECONDARY_CONFOUNDER = "sex"
CONFOUNDER_COLS = [SECONDARY_CONFOUNDER, PRIMARY_CONFOUNDER]
N_PCS_KS        = 5

np.random.seed(SEED)

In [2]:
# ------------------------------- Helpers ------------------------------------
def load_vst(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    if df.shape[0] == 155 and df.shape[1] != 155:
        df = df.T
    return df  # genes x samples


def align_metadata(meta: pd.DataFrame, sample_ids) -> pd.DataFrame:
    meta = meta.copy()
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    if CONDITION_COL not in meta.columns and "Class_Label" in meta.columns:
        meta = meta.rename(columns={"Class_Label": CONDITION_COL})
    needed = [CONDITION_COL] + CONFOUNDER_COLS
    missing = [c for c in needed if c not in meta.columns]
    if missing:
        raise KeyError(f"Metadata missing required columns: {missing}")
    return meta.loc[sample_ids, needed]


def build_strat_key(meta):
    """
    Build the strongest viable stratification key.

    Strong control on cell_type means cell_type is anchored on every level
    of the chain. The fallback only relaxes the *secondary* (sex) layer or,
    in the worst case, drops cell_type entirely if even (condition x cell_type)
    has singletons (which would be a pathological cohort and is unexpected here).
    """
    key1 = (meta[CONDITION_COL].astype(str)
            + "__" + meta[PRIMARY_CONFOUNDER].astype(str)
            + "__" + meta[SECONDARY_CONFOUNDER].astype(str))
    key2 = (meta[CONDITION_COL].astype(str)
            + "__" + meta[PRIMARY_CONFOUNDER].astype(str))
    key3 = meta[CONDITION_COL].astype(str)
    if (key1.value_counts() >= 2).all():
        return key1, "condition x cell_type x sex"
    if (key2.value_counts() >= 2).all():
        return key2, "condition x cell_type (sex relaxed)"
    return key3, "condition only (cell_type stratification not possible)"


def compute_chi2(meta_all, meta_train, meta_test):
    """Marginal chi-squared per stratification variable (train vs test)."""
    variables = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                                   if c in meta_all.columns]
    rows = {}
    for var in variables:
        cats = sorted(meta_all[var].unique())
        ct = pd.DataFrame({
            "train": meta_train[var].value_counts().reindex(cats, fill_value=0),
            "test":  meta_test[var].value_counts().reindex(cats, fill_value=0),
        })
        chi2, p, dof, _ = chi2_contingency(ct)
        rows[var] = {"chi2": chi2, "p": p, "dof": int(dof)}
    return rows


def compute_chi2_per_celltype(meta_train, meta_test):
    """Per-cell_type chi-squared on (train vs test) condition distribution.

    A composite-marginal chi-squared can pass even when condition ratios drift
    *within* a cell_type. This per-stratum check makes such drift visible.
    """
    cell_types = sorted(set(meta_train[PRIMARY_CONFOUNDER]) |
                        set(meta_test[PRIMARY_CONFOUNDER]))
    rows = []
    for ct_ in cell_types:
        tr = meta_train[meta_train[PRIMARY_CONFOUNDER] == ct_]
        te = meta_test [meta_test [PRIMARY_CONFOUNDER] == ct_]
        cats = sorted(set(tr[CONDITION_COL]) | set(te[CONDITION_COL]))
        if not cats or tr.empty or te.empty:
            rows.append({"cell_type": ct_, "n_train": len(tr), "n_test": len(te),
                         "chi2": np.nan, "dof": 0, "p_value": np.nan,
                         "balanced_at_0.05": False,
                         "note": "empty side"})
            continue
        tab = pd.DataFrame({
            "train": tr[CONDITION_COL].value_counts().reindex(cats, fill_value=0),
            "test":  te[CONDITION_COL].value_counts().reindex(cats, fill_value=0),
        })
        try:
            chi2, p, dof, _ = chi2_contingency(tab)
        except ValueError:
            chi2, p, dof = np.nan, np.nan, 0
        rows.append({"cell_type": ct_, "n_train": len(tr), "n_test": len(te),
                     "chi2": chi2, "dof": int(dof), "p_value": p,
                     "balanced_at_0.05": (p is not np.nan and p > 0.05),
                     "note": ""})
    return pd.DataFrame(rows)


def save_chi2_csv(chi2_results, out_path):
    pd.DataFrame([
        {"variable": v, "chi2": r["chi2"], "dof": r["dof"],
         "p_value": r["p"], "balanced_at_0.05": r["p"] > 0.05}
        for v, r in chi2_results.items()
    ]).to_csv(out_path, index=False)

In [3]:
# --------------------------- Stratification plot ----------------------------
def plot_stratification(meta_all, meta_train, meta_test,
                        chi2_results, ct_chi2, out_path):
    """Composite figure: marginals, composite stacks, summary, cross-tabs."""
    variables = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                                   if c in meta_all.columns]
    confounders = [c for c in CONFOUNDER_COLS if c in meta_all.columns]

    fig = plt.figure(figsize=(20, 17))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38)

    palette = {"Overall": "#95a5a6", "Train": "#2ecc71", "Test": "#e74c3c"}
    splits  = {"Overall": meta_all, "Train": meta_train, "Test": meta_test}

    # Row 1: marginal proportions, with chi^2 in the title
    for col_idx, var in enumerate(variables[:3]):
        ax = fig.add_subplot(gs[0, col_idx])
        cats = sorted(meta_all[var].unique())
        x = np.arange(len(cats)); width = 0.25
        offsets = [-width, 0, width]
        for s_idx, (split_name, m_s) in enumerate(splits.items()):
            counts = m_s[var].value_counts().reindex(cats, fill_value=0)
            props = counts / max(1, len(m_s)) * 100
            bars = ax.bar(x + offsets[s_idx], props.values, width,
                          label=f"{split_name} (n={len(m_s)})",
                          color=palette[split_name],
                          edgecolor="white", linewidth=0.5, alpha=0.88)
            for bar, cnt in zip(bars, counts.values):
                if bar.get_height() > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() / 2, f"n={cnt}",
                            ha="center", va="center",
                            fontsize=6.5, color="white", fontweight="bold")
        r = chi2_results[var]
        sig = (f"OK  p={r['p']:.3f}" if r["p"] > 0.05
               else f"WARN p={r['p']:.3f}")
        ax.set_title(f"{var.replace('_', ' ').title()}\n"
                     f"Chi^2 train vs test: {sig}",
                     fontsize=10, fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=9)
        ax.set_ylabel("Proportion (%)", fontsize=9)
        ax.set_ylim(0, max(ax.get_ylim()[1] * 1.15, 15))
        ax.legend(fontsize=7.5, loc="upper right")
        ax.grid(axis="y", alpha=0.25, linestyle=":"); sns.despine(ax=ax)

    # Row 2 panels 0,1: condition x confounder stacked bars
    for conf_idx, conf_var in enumerate(confounders[:2]):
        ax = fig.add_subplot(gs[1, conf_idx])
        cond_cats = sorted(meta_all[CONDITION_COL].unique())
        conf_cats = sorted(meta_all[conf_var].unique())
        cmap = dict(zip(conf_cats, sns.color_palette("Set2", len(conf_cats))))
        x_ticks, x_labels, current_x = [], [], 0
        for split_name, m_s in [("Train", meta_train), ("Test", meta_test)]:
            for cond in cond_cats:
                sub = m_s[m_s[CONDITION_COL] == cond]
                bottom = 0
                for cv in conf_cats:
                    cnt = (sub[conf_var] == cv).sum()
                    prop = cnt / len(sub) * 100 if len(sub) > 0 else 0
                    ax.bar(current_x, prop, 1, bottom=bottom,
                           color=cmap[cv], edgecolor="white",
                           linewidth=0.4, alpha=0.88)
                    if prop > 8:
                        ax.text(current_x, bottom + prop / 2,
                                f"{prop:.0f}%", ha="center", va="center",
                                fontsize=6, color="white", fontweight="bold")
                    bottom += prop
                x_ticks.append(current_x)
                x_labels.append(f"{cond}\n({split_name})")
                current_x += 1
            current_x += 0.6
        handles = [plt.Rectangle((0, 0), 1, 1, color=cmap[c], label=c)
                   for c in conf_cats]
        ax.legend(handles=handles, title=conf_var.replace("_", " ").title(),
                  fontsize=7.5, title_fontsize=8, loc="lower right")
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, fontsize=7.5)
        ax.set_ylabel("Proportion within condition (%)", fontsize=9)
        ax.set_ylim(0, 115)
        ax.set_title(f"Condition x {conf_var.replace('_', ' ').title()}\n"
                     f"(stacked proportions)",
                     fontsize=10, fontweight="bold")
        ax.grid(axis="y", alpha=0.25, linestyle=":"); sns.despine(ax=ax)

    # Row 2 panel 2: combined chi^2 summary table (marginal + per-cell_type)
    ax_tbl = fig.add_subplot(gs[1, 2]); ax_tbl.axis("off")
    rows = [["Variable / cell_type", "Chi^2", "df", "p", "Balanced?"]]
    for var in variables:
        r = chi2_results[var]
        rows.append([var.replace("_", " ").title(),
                     f"{r['chi2']:.3f}", str(r["dof"]),
                     f"{r['p']:.4f}",
                     "Yes" if r["p"] > 0.05 else "WARN"])
    rows.append(["--- per cell_type (cond) ---", "", "", "", ""])
    for _, r in ct_chi2.iterrows():
        ok = "Yes" if r["balanced_at_0.05"] else "WARN"
        rows.append([f"  {r['cell_type']} (Tr={r['n_train']}, Te={r['n_test']})",
                     f"{r['chi2']:.3f}" if pd.notna(r['chi2']) else "-",
                     str(r['dof']),
                     f"{r['p_value']:.4f}" if pd.notna(r['p_value']) else "-",
                     ok])
    tbl = ax_tbl.table(cellText=rows[1:], colLabels=rows[0],
                       cellLoc="center", loc="center",
                       bbox=[0, 0.05, 1, 0.92])
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    for (rr, cc), cell in tbl.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if rr == 0:
            cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white", fontweight="bold")
        elif "Yes" in cell.get_text().get_text():
            cell.set_facecolor("#d5f5e3")
        elif "WARN" in cell.get_text().get_text():
            cell.set_facecolor("#fadbd8")
        else:
            cell.set_facecolor("#f8f9fa" if rr % 2 == 0 else "white")
    ax_tbl.set_title("Chi-squared: marginal + per cell_type\n"
                     "(H0: train and test condition distributions match)",
                     fontsize=10, fontweight="bold", pad=12)

    # Row 3: cross-tab heatmaps
    pairs = [(CONDITION_COL, v) for v in confounders[:2]]
    splits_3 = [("Train", meta_train), ("Test", meta_test)]
    positions = [(2, 0), (2, 1), (2, 2)]; h_idx = 0
    for row_var, col_var in pairs:
        for split_name, m_s in splits_3:
            if h_idx >= len(positions): break
            ax = fig.add_subplot(gs[positions[h_idx]])
            ct = pd.crosstab(m_s[row_var], m_s[col_var])
            sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax,
                        linewidths=0.5, linecolor="white",
                        cbar_kws={"shrink": 0.75})
            ax.set_title(f"{row_var.replace('_', ' ').title()} x "
                         f"{col_var.replace('_', ' ').title()}\n"
                         f"({split_name})",
                         fontsize=9.5, fontweight="bold")
            ax.set_xlabel(col_var.replace("_", " ").title(), fontsize=8.5)
            ax.set_ylabel(row_var.replace("_", " ").title(), fontsize=8.5)
            ax.tick_params(labelsize=8); h_idx += 1

    fig.suptitle(
        "Stage 2 (PD vs HC) - Composite stratification with strong cell_type control\n"
        f"(70/30 split; primary anchor = {PRIMARY_CONFOUNDER}, secondary = {SECONDARY_CONFOUNDER})",
        fontsize=13, fontweight="bold", y=1.01)
    plt.savefig(out_path, dpi=200, bbox_inches="tight"); plt.close()

In [4]:
# --------------------------- Variance partition -----------------------------
def variance_partition(Y, meta_block):
    """Per-gene partial R^2 for each predictor in meta_block."""
    cols = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                              if c in meta_block.columns]
    enc = OneHotEncoder(drop="first", sparse_output=False,
                        handle_unknown="ignore")
    blocks, parts, cursor = {}, [np.ones((len(meta_block), 1))], 1
    for col in cols:
        d = enc.fit_transform(meta_block[[col]])
        blocks[col] = (cursor, cursor + d.shape[1])
        parts.append(d); cursor += d.shape[1]
    X = np.hstack(parts)

    Y_mean = Y.mean(axis=0, keepdims=True)
    SS_tot = ((Y - Y_mean) ** 2).sum(axis=0) + 1e-10

    def ss_res(Xd):
        B = np.linalg.lstsq(Xd, Y, rcond=None)[0]
        return ((Y - Xd @ B) ** 2).sum(axis=0)

    SS_full = ss_res(X)
    out = {}
    for col in cols:
        s, e = blocks[col]
        Xr = np.hstack([X[:, :s], X[:, e:]])
        out[col] = np.clip((ss_res(Xr) - SS_full) / SS_tot, 0, 1)
    out["residual"] = np.clip(SS_full / SS_tot, 0, 1)
    return out


def fit_mirror_ols(X_train, meta_train, conf_cols):
    """Fit OLS X ~ confounders on TRAIN; return a closure that residualizes
    any (X, meta) pair using the same encoder + betas (no test refit)."""
    enc = OneHotEncoder(drop="first", sparse_output=False,
                        handle_unknown="ignore")
    enc.fit(meta_train[conf_cols])

    def design_matrix(meta_block):
        D = enc.transform(meta_block[conf_cols])
        return np.column_stack([np.ones(len(meta_block)), D])

    C_train = design_matrix(meta_train)
    B_ols = np.linalg.pinv(C_train) @ X_train

    def residualize(X, meta_block):
        C = design_matrix(meta_block)
        return X - C[:, 1:] @ B_ols[1:, :]

    return residualize, B_ols, enc


def plot_variance_partitioning(r2_before, r2_after, n_genes, out_path):
    predictors = [CONDITION_COL] + [c for c in CONFOUNDER_COLS
                                    if c in r2_before]
    cols = predictors + ["residual"]; n_pred = len(cols)
    color_map = {
        CONDITION_COL: "#2980b9",
        "sex":         "#e74c3c",
        "cell_type":   "#e67e22",
        "residual":    "#7f8c8d",
    }
    fig, axes = plt.subplots(2, n_pred, figsize=(3.5 * n_pred, 7),
                             sharey=True, constrained_layout=True)
    row_labels = ["Before residualization", "After residualization"]
    for row, (r2_dict, lbl) in enumerate(zip([r2_before, r2_after], row_labels)):
        for col, var in enumerate(cols):
            ax = axes[row, col]
            data = r2_dict.get(var, np.zeros(n_genes)) * 100
            color = color_map.get(var, "#95a5a6")
            parts = ax.violinplot(data, positions=[0],
                                  showmedians=True, showextrema=True, widths=0.7)
            for pc in parts["bodies"]:
                pc.set_facecolor(color); pc.set_alpha(0.55)
            parts["cmedians"].set_color("black")
            parts["cmedians"].set_linewidth(2)
            med = float(np.median(data))
            ax.text(0.08, med, f" {med:.1f}%", va="center",
                    fontsize=8.5, color="black", fontweight="bold",
                    transform=ax.get_yaxis_transform())
            title = (var.replace("_", " ").title()
                     if var != "residual" else "Unexplained")
            if row == 0:
                ax.set_title(title, fontsize=10, fontweight="bold", color=color)
            ax.set_xticks([])
            if col == 0:
                ax.set_ylabel(f"{lbl}\nPartial R^2 (%)", fontsize=8.5)
            ax.set_ylim(-2, 105)
            ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
            sns.despine(ax=ax)
    fig.suptitle(
        "Variance Partitioning (PD vs HC) - Per-gene Partial R^2 "
        "Before vs After Mirror OLS\n"
        "(cell_type R^2 should drop to ~0; condition R^2 should be preserved)",
        fontsize=11, fontweight="bold")
    plt.savefig(out_path, dpi=200, bbox_inches="tight"); plt.close()

In [5]:
# ----------------------------- Main pipeline --------------------------------
def main():
    print("=" * 70)
    print("STAGE 2 (PD vs HC, stratum-aware) - 70/30 split with strong cell_type control")
    print("=" * 70)

    vst = load_vst(INPUT_VST)
    sample_ids = vst.columns.tolist()
    gene_names = vst.index.tolist()
    X_full = vst.T.values

    meta = align_metadata(pd.read_csv(METADATA_PATH), sample_ids)
    y = meta[CONDITION_COL].values

    strat_key, strat_label = build_strat_key(meta)
    counts = strat_key.value_counts()
    print(f"\nStratification key: {strat_label}")
    print(f"Strata (n={len(counts)}):\n{counts.to_string()}")

    sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE,
                                 random_state=SEED)
    train_idx, test_idx = next(sss.split(X_full, strat_key))

    X_train = X_full[train_idx]; X_test = X_full[test_idx]
    y_train = y[train_idx];      y_test = y[test_idx]
    meta_train = meta.iloc[train_idx].copy()
    meta_test  = meta.iloc[test_idx].copy()
    train_ids = [sample_ids[i] for i in train_idx]
    test_ids  = [sample_ids[i] for i in test_idx]

    print(f"\nTrain (n={len(y_train)}):\n{pd.Series(y_train).value_counts().to_string()}")
    print(f"\nTest  (n={len(y_test)}):\n{pd.Series(y_test).value_counts().to_string()}")

    # ---- (a) marginal chi-squared ----
    chi2_results = compute_chi2(meta, meta_train, meta_test)
    save_chi2_csv(chi2_results, OUTPUT_DIR / "chi_squared.csv")
    print("\nChi-squared (marginal, train vs test):")
    for v, r in chi2_results.items():
        flag = "OK" if r["p"] > 0.05 else "WARN"
        print(f"  {v:12s} chi2={r['chi2']:.3f} df={r['dof']} "
              f"p={r['p']:.4f}  [{flag}]")

    # ---- (a') per-cell_type chi-squared (strong cell_type control) ----
    ct_chi2 = compute_chi2_per_celltype(meta_train, meta_test)
    ct_chi2.to_csv(OUTPUT_DIR / "chi_squared_per_cell_type.csv", index=False)
    print("\nChi-squared (per cell_type, condition train vs test):")
    for _, r in ct_chi2.iterrows():
        flag = "OK" if r["balanced_at_0.05"] else "WARN"
        p_str = f"p={r['p_value']:.4f}" if pd.notna(r['p_value']) else "p=NA"
        print(f"  {r['cell_type']:5s} Tr={int(r['n_train']):2d} Te={int(r['n_test']):2d} "
              f"chi2={r['chi2'] if pd.notna(r['chi2']) else 'NA':>6} "
              f"{p_str}  [{flag}]")

    plot_stratification(meta, meta_train, meta_test, chi2_results, ct_chi2,
                        OUTPUT_DIR / "stratification.png")

    # ---- (b) KS-test on TRAIN-fit PCs ----
    n_pcs = min(N_PCS_KS, min(X_train.shape) - 1)
    pca = PCA(n_components=n_pcs, random_state=SEED); pca.fit(X_train)
    pcs_train = pca.transform(X_train); pcs_test = pca.transform(X_test)

    ks_rows = []
    for i in range(n_pcs):
        ks, p = ks_2samp(pcs_train[:, i], pcs_test[:, i])
        ks_rows.append({"PC": f"PC{i + 1}",
                        "var_explained": pca.explained_variance_ratio_[i],
                        "KS_stat": ks, "p_value": p,
                        "balanced_at_0.05": p > 0.05})
    pd.DataFrame(ks_rows).to_csv(OUTPUT_DIR / "ks_test_on_pcs.csv", index=False)
    print(f"\nKS-test on top-{n_pcs} TRAIN-fit PCs (train vs test):")
    for r in ks_rows:
        flag = "OK" if r["balanced_at_0.05"] else "WARN"
        print(f"  {r['PC']}: var_exp={r['var_explained']:.1%}, "
              f"KS={r['KS_stat']:.4f}, p={r['p_value']:.4f}  [{flag}]")

    # ---- (c) Variance partitioning before vs after mirror OLS ----
    conf_cols = [c for c in CONFOUNDER_COLS if c in meta_train.columns]
    print(f"\nFitting mirror OLS on TRAIN (confounders = {conf_cols}) ...")
    residualize, B_ols, _enc = fit_mirror_ols(X_train, meta_train, conf_cols)
    X_train_resid = residualize(X_train, meta_train)
    X_test_resid  = residualize(X_test,  meta_test)

    print("Computing per-gene partial R^2 (before vs after) ...")
    r2_before = variance_partition(X_train,       meta_train)
    r2_after  = variance_partition(X_train_resid, meta_train)

    r2_table = pd.DataFrame({"gene": gene_names})
    for k, v in r2_before.items(): r2_table[f"{k}_before"] = v
    for k, v in r2_after.items():  r2_table[f"{k}_after"]  = v
    r2_table.to_csv(OUTPUT_DIR / "variance_partition_per_gene.csv", index=False)

    plot_variance_partitioning(r2_before, r2_after, X_train.shape[1],
                               OUTPUT_DIR / "variance_partitioning.png")

    print("Median partial R^2 (% across genes):")
    for var in [CONDITION_COL] + conf_cols + ["residual"]:
        before = float(np.median(r2_before[var])) * 100
        after  = float(np.median(r2_after[var]))  * 100
        print(f"  {var:12s} before={before:5.1f}%   after={after:5.1f}%")

    # ---- Save split + residualized data ----
    print("\nSaving outputs ...")
    meta_train.to_csv(OUTPUT_DIR / "meta_train.csv")
    meta_test.to_csv (OUTPUT_DIR / "meta_test.csv")

    train_df = pd.DataFrame(X_train.T, index=gene_names, columns=train_ids)
    train_df.index.name = "gene"; train_df.to_csv(OUTPUT_DIR / "train.csv")
    test_df  = pd.DataFrame(X_test.T,  index=gene_names, columns=test_ids)
    test_df.index.name  = "gene"; test_df.to_csv (OUTPUT_DIR / "test.csv")

    train_resid_df = pd.DataFrame(X_train_resid.T,
                                   index=gene_names, columns=train_ids)
    train_resid_df.index.name = "gene"
    train_resid_df.to_csv(OUTPUT_DIR / "train_residualized.csv")
    test_resid_df  = pd.DataFrame(X_test_resid.T,
                                   index=gene_names, columns=test_ids)
    test_resid_df.index.name = "gene"
    test_resid_df.to_csv (OUTPUT_DIR / "test_residualized.csv")

    pd.DataFrame(B_ols, columns=gene_names).to_csv(
        OUTPUT_DIR / "mirror_ols_betas.csv", index_label="design_row")

    rows = []
    for split_name, m_s in [("Overall", meta), ("Train", meta_train),
                            ("Test", meta_test)]:
        for var in [CONDITION_COL] + CONFOUNDER_COLS:
            counts_v = m_s[var].value_counts().sort_index()
            for cat, cnt in counts_v.items():
                rows.append({"split": split_name, "variable": var,
                             "category": cat, "count": int(cnt),
                             "total": len(m_s),
                             "proportion": round(cnt / len(m_s) * 100, 2)})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

    print(f"\nAll outputs in: {OUTPUT_DIR}")
    print(" - stratification.png                  diagnostic figure (incl. per-cell_type chi^2)")
    print(" - chi_squared.csv                     marginal chi-squared per variable")
    print(" - chi_squared_per_cell_type.csv       per-cell_type chi-squared (strong control)")
    print(" - ks_test_on_pcs.csv                  KS test on TRAIN-fit PCs")
    print(" - variance_partitioning.png           partial R^2 violin plot")
    print(" - variance_partition_per_gene.csv     partial R^2 per gene")
    print(" - mirror_ols_betas.csv                TRAIN-fit OLS coefficients")
    print(" - train.csv / test.csv                raw VST splits")
    print(" - train_residualized.csv / test_residualized.csv  mirror-OLS residuals")
    print(" - meta_train.csv / meta_test.csv      metadata for each split")
    print(" - split_summary.csv                   counts and proportions")

main()

STAGE 2 (PD vs HC, stratum-aware) - 70/30 split with strong cell_type control

Stratification key: condition x cell_type x sex
Strata (n=12):
PD__CD8__M     29
PD__CD4__M     27
PD__PBMC__M    26
HC__CD8__F     15
HC__CD4__F     13
HC__PBMC__F    10
PD__CD4__F      8
PD__CD8__F      7
PD__PBMC__F     5
HC__PBMC__M     5
HC__CD4__M      5
HC__CD8__M      5

Train (n=108):
PD    72
HC    36

Test  (n=47):
PD    30
HC    17

Chi-squared (marginal, train vs test):
  condition    chi2=0.025 df=1 p=0.8744  [OK]
  sex          chi2=0.001 df=1 p=0.9749  [OK]
  cell_type    chi2=0.001 df=2 p=0.9996  [OK]

Chi-squared (per cell_type, condition train vs test):
  CD4   Tr=37 Te=16 chi2=0.0017408033033033047 p=0.9667  [OK]
  CD8   Tr=39 Te=17 chi2=   0.0 p=1.0000  [OK]
  PBMC  Tr=32 Te=14 chi2=   0.0 p=1.0000  [OK]

KS-test on top-5 TRAIN-fit PCs (train vs test):
  PC1: var_exp=16.8%, KS=0.1708, p=0.2577  [OK]
  PC2: var_exp=14.0%, KS=0.1123, p=0.7512  [OK]
  PC3: var_exp=6.5%, KS=0.1468, p=0.4300 